<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/TravelAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import asyncio
import random
import json
from datetime import datetime
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Callable, Any
from abc import ABC, abstractmethod
import uuid
import nest_asyncio

# ==================== DATA MODELS ====================

class BookingStatus(Enum):
    PENDING = auto()
    CHECKING_AVAILABILITY = auto()
    SELECTING_SEATS = auto()
    PAYMENT_PENDING = auto()
    CONFIRMED = auto()
    FAILED = auto()
    CANCELLED = auto()

@dataclass
class Seat:
    row: str
    number: int
    category: str  # SILVER, GOLD, PLATINUM
    price: float
    is_available: bool = True
    is_selected: bool = False

    def __repr__(self):
        status = "🟢" if self.is_available else "🔴"
        if self.is_selected:
            status = "🟡"
        return f"{status} {self.row}{self.number}({self.category})"

@dataclass
class Movie:
    id: str
    title: str
    language: str
    format: str  # 2D, 3D, IMAX
    duration: int  # minutes
    rating: float

@dataclass
class Show:
    id: str
    movie: Movie
    theater: str
    screen: str
    start_time: datetime
    seats: Dict[str, Seat] = field(default_factory=dict)  # key: "A1", "B2" etc

    def get_available_seats(self, category: Optional[str] = None) -> List[Seat]:
        seats = [s for s in self.seats.values() if s.is_available and not s.is_selected]
        if category:
            seats = [s for s in seats if s.category == category]
        return seats

@dataclass
class BookingRequest:
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    movie_title: str = ""
    theater: str = ""
    show_time: Optional[datetime] = None
    num_tickets: int = 1
    preferred_category: str = "GOLD"
    status: BookingStatus = BookingStatus.PENDING
    selected_seats: List[Seat] = field(default_factory=list)
    total_amount: float = 0.0
    payment_method: str = ""
    user_email: str = ""
    error_message: Optional[str] = None
    audit_log: List[str] = field(default_factory=list)

    def log(self, message: str):
        timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        self.audit_log.append(f"[{timestamp}] {message}")

# ==================== MESSAGE BUS / EVENT SYSTEM ====================

class EventType(Enum):
    CHECK_AVAILABILITY = "check_availability"
    AVAILABILITY_RESULT = "availability_result"
    SELECT_SEATS = "select_seats"
    SEATS_SELECTED = "seats_selected"
    PROCESS_PAYMENT = "process_payment"
    PAYMENT_RESULT = "payment_result"
    BOOKING_COMPLETE = "booking_complete"
    BOOKING_FAILED = "booking_failed"
    RELEASE_SEATS = "release_seats"

@dataclass
class Event:
    type: EventType
    booking_id: str
    payload: Dict[str, Any] = field(default_factory=dict)
    source: str = "system"

class MessageBus:
    """Async event bus for inter-agent communication"""
    def __init__(self):
        self.subscribers: Dict[EventType, List[Callable]] = {et: [] for et in EventType}
        self.event_history: List[Event] = []

    def subscribe(self, event_type: EventType, callback: Callable):
        self.subscribers[event_type].append(callback)

    async def publish(self, event: Event):
        self.event_history.append(event)
        print(f"\n📡 EVENT: {event.type.value} | Booking: {event.booking_id} | From: {event.source}")
        for callback in self.subscribers[event.type]: # Changed event_type to event.type
            try:
                if asyncio.iscoroutinefunction(callback):
                    await callback(event)
                else:
                    callback(event)
            except Exception as e:
                print(f"❌ Error handling event {event.type.value}: {e}")

# ==================== AGENT BASE CLASS ====================

class Agent(ABC):
    def __init__(self, name: str, bus: MessageBus):
        self.name = name
        self.bus = bus
        self.active_bookings: Dict[str, BookingRequest] = {}

    def log(self, message: str):
        print(f"  🤖 [{self.name}] {message}")

    @abstractmethod
    async def handle_event(self, event: Event):
        pass

# ==================== 1. AVAILABILITY AGENT ====================

class AvailabilityAgent(Agent):
    """Checks movie/show availability and seat inventory"""

    def __init__(self, bus: MessageBus, shows_db: Dict[str, Show]):
        super().__init__("AvailabilityAgent", bus)
        self.shows_db = shows_db
        self.bus.subscribe(EventType.CHECK_AVAILABILITY, self.handle_event)

    async def handle_event(self, event: Event):
        request = BookingRequest(**event.payload['request'])
        request.status = BookingStatus.CHECKING_AVAILABILITY
        request.log("Availability check initiated")
        self.active_bookings[request.id] = request

        self.log(f"Checking availability for '{request.movie_title}' at {request.theater}")

        # Simulate API delay
        await asyncio.sleep(random.uniform(0.5, 1.5))

        # Find matching show
        matching_shows = [
            s for s in self.shows_db.values()
            if s.movie.title.lower() == request.movie_title.lower()
            and s.theater.lower() == request.theater.lower()
        ]

        if not matching_shows:
            request.status = BookingStatus.FAILED
            request.error_message = "Show not found"
            request.log("Availability check FAILED - Show not found")
            await self.bus.publish(Event(EventType.BOOKING_FAILED, request.id, {
                'request': request.__dict__,
                'error': request.error_message
            }, self.name))
            return

        show = matching_shows[0]
        available = show.get_available_seats(request.preferred_category)

        if len(available) < request.num_tickets:
            request.status = BookingStatus.FAILED
            request.error_message = f"Only {len(available)} seats available in {request.preferred_category}"
            request.log(f"Availability check FAILED - Insufficient seats")
            await self.bus.publish(Event(EventType.BOOKING_FAILED, request.id, {
                'request': request.__dict__,
                'error': request.error_message
            }, self.name))
            return

        request.log(f"Availability check PASSED - {len(available)} seats available")
        self.log(f"✅ Found {len(available)} available {request.preferred_category} seats")

        await self.bus.publish(Event(EventType.AVAILABILITY_RESULT, request.id, {
            'request': request.__dict__,
            'show_id': show.id,
            'available_seats': [f"{s.row}{s.number}" for s in available[:10]]  # Send top 10
        }, self.name))

# ==================== 2. SEAT SELECTION AGENT ====================

class SeatSelectionAgent(Agent):
    """Handles intelligent seat selection and blocking"""

    def __init__(self, bus: MessageBus, shows_db: Dict[str, Show]):
        super().__init__("SeatSelectionAgent", bus)
        self.shows_db = shows_db
        self.bus.subscribe(EventType.SELECT_SEATS, self.handle_event)
        self.bus.subscribe(EventType.AVAILABILITY_RESULT, self.auto_select_seats)
        self.locked_seats: Dict[str, List[str]] = {}  # booking_id -> seat_keys

    async def auto_select_seats(self, event: Event):
        """Automatically picks best available seats when availability is confirmed"""
        request = BookingRequest(**event.payload['request'])
        show_id = event.payload['show_id']
        show = self.shows_db[show_id]

        request.status = BookingStatus.SELECTING_SEATS
        request.log("Auto-selecting best seats")
        self.log(f"Selecting {request.num_tickets} best seats for booking {request.id}")

        await asyncio.sleep(random.uniform(0.3, 0.8))  # Processing delay

        available = show.get_available_seats(request.preferred_category)

        # Intelligent selection: prefer consecutive seats, middle rows
        selected = self._find_best_consecutive_seats(available, request.num_tickets)

        if len(selected) < request.num_tickets:
            # Fallback to any available
            selected = available[:request.num_tickets]

        # Lock seats (mark as selected)
        for seat in selected:
            seat.is_selected = True
            key = f"{seat.row}{seat.number}"
            if request.id not in self.locked_seats:
                self.locked_seats[request.id] = []
            self.locked_seats[request.id].append(key)

        request.selected_seats = selected
        request.total_amount = sum(s.price for s in selected)

        request.log(f"Selected seats: {[f'{s.row}{s.number}' for s in selected]}")
        request.log(f"Total amount: ₹{request.total_amount}")
        self.log(f"✅ Locked seats: {[f'{s.row}{s.number}' for s in selected]} | Total: ₹{request.total_amount}")

        # Start seat hold timer (simulate 10-min hold)
        asyncio.create_task(self._seat_hold_timer(request.id, show_id))

        await self.bus.publish(Event(EventType.SEATS_SELECTED, request.id, {
            'request': request.__dict__,
            'show_id': show_id,
            'selected_seats': [f"{s.row}{s.number}" for s in selected]
        }, self.name))

    def _find_best_consecutive_seats(self, seats: List[Seat], count: int) -> List[Seat]:
        """Find consecutive seats in same row"""
        row_groups: Dict[str, List[Seat]] = {}
        for seat in seats:
            row_groups.setdefault(seat.row, []).append(seat)

        for row in sorted(row_groups.keys()):
            row_seats = sorted(row_groups[row], key=lambda s: s.number)
            for i in range(len(row_seats) - count + 1):
                if all(row_seats[j].number == row_seats[i].number + (j - i)
                       for j in range(i, i + count)):
                    return row_seats[i:i + count]
        return []

    async def _seat_hold_timer(self, booking_id: str, show_id: str):
        """Release seats if booking not completed within timeout"""
        await asyncio.sleep(15)  # 15 seconds for demo (real: 10 mins)
        if booking_id in self.locked_seats:
            self.log(f"⏰ Seat hold expired for booking {booking_id}, releasing seats")
            await self.bus.publish(Event(EventType.RELEASE_SEATS, booking_id, {
                'show_id': show_id,
                'seats': self.locked_seats[booking_id]
            }, self.name))

    async def handle_event(self, event: Event):
        # Handle manual seat selection if needed
        pass

# ==================== 3. PAYMENT AGENT ====================

class PaymentAgent(Agent):
    """Handles payment processing, validation, and refunds"""

    def __init__(self, bus: MessageBus):
        super().__init__("PaymentAgent", bus)
        self.bus.subscribe(EventType.PROCESS_PAYMENT, self.handle_event)
        self.bus.subscribe(EventType.SEATS_SELECTED, self.auto_process_payment)
        self.success_rate = 0.85  # 85% success rate for simulation

    async def auto_process_payment(self, event: Event):
        """Automatically triggers payment after seat selection"""
        request = BookingRequest(**event.payload['request'])
        request.status = BookingStatus.PAYMENT_PENDING
        request.log("Auto-initiating payment")
        self.log(f"Processing payment of ₹{request.total_amount} for booking {request.id}")

        await asyncio.sleep(random.uniform(1.0, 2.0))  # Payment gateway delay

        # Simulate payment processing
        success = random.random() < self.success_rate

        if success:
            transaction_id = f"TXN{uuid.uuid4().hex[:12].upper()}"
            request.log(f"Payment SUCCESSFUL - Transaction ID: {transaction_id}")
            self.log(f"✅ Payment confirmed | TXN: {transaction_id}")

            await self.bus.publish(Event(EventType.PAYMENT_RESULT, request.id, {
                'request': request.__dict__,
                'success': True,
                'transaction_id': transaction_id,
                'amount_paid': request.total_amount
            }, self.name))
        else:
            error = random.choice([
                "Insufficient funds",
                "Bank timeout",
                "Card declined",
                "UPI transaction failed"
            ])
            request.log(f"Payment FAILED - {error}")
            self.log(f"❌ Payment failed: {error}")

            await self.bus.publish(Event(EventType.PAYMENT_RESULT, request.id, {
                'request': request.__dict__,
                'success': False,
                'error': error
            }, self.name))

    async def handle_event(self, event: Event):
        # Handle manual payment triggers
        pass

# ==================== 4. BOOKING ORCHESTRATOR AGENT ====================

class BookingOrchestrator(Agent):
    """Central coordinator that manages the booking lifecycle"""

    def __init__(self, bus: MessageBus, shows_db: Dict[str, Show]):
        super().__init__("BookingOrchestrator", bus)
        self.shows_db = shows_db
        self.completed_bookings: Dict[str, BookingRequest] = {}
        self.failed_bookings: Dict[str, BookingRequest] = {}

        # Subscribe to all completion events
        self.bus.subscribe(EventType.PAYMENT_RESULT, self.handle_payment_result)
        self.bus.subscribe(EventType.BOOKING_FAILED, self.handle_failure)
        self.bus.subscribe(EventType.RELEASE_SEATS, self.handle_release)

    async def start_booking(self, request: BookingRequest) -> str:
        """Initiate a new booking workflow"""
        self.log(f"Starting new booking flow: {request.id}")
        request.log("Booking flow initiated by orchestrator")
        self.active_bookings[request.id] = request

        # Step 1: Check Availability
        await self.bus.publish(Event(EventType.CHECK_AVAILABILITY, request.id, {
            'request': request.__dict__
        }, self.name))

        return request.id

    async def handle_payment_result(self, event: Event):
        """Handle payment completion"""
        payload = event.payload
        request = BookingRequest(**payload['request'])

        if payload['success']:
            request.status = BookingStatus.CONFIRMED
            request.log("Booking CONFIRMED")
            self.log(f"🎉 Booking {request.id} CONFIRMED!")

            # Mark seats as booked (not available)
            show_id = event.payload.get('show_id', '')
            for seat in request.selected_seats:
                seat.is_available = False
                seat.is_selected = False

            self.completed_bookings[request.id] = request

            await self.bus.publish(Event(EventType.BOOKING_COMPLETE, request.id, {
                'request': request.__dict__,
                'transaction_id': payload['transaction_id']
            }, self.name))
        else:
            request.status = BookingStatus.FAILED
            request.error_message = f"Payment failed: {payload['error']}"
            request.log(f"Booking FAILED - {request.error_message}")
            self.failed_bookings[request.id] = request

            # Release held seats
            await self.bus.publish(Event(EventType.RELEASE_SEATS, request.id, {
                'seats': [f"{s.row}{s.number}" for s in request.selected_seats]
            }, self.name))

    async def handle_failure(self, event: Event):
        """Handle booking failures"""
        request = BookingRequest(**event.payload['request'])
        self.failed_bookings[request.id] = request
        self.log(f"⚠️ Booking {request.id} failed: {request.error_message}")

    async def handle_release(self, event: Event):
        """Release held seats back to inventory"""
        seats_to_release = event.payload.get('seats', [])
        show_id = event.payload.get('show_id', '')

        # Find show and release seats
        for show in self.shows_db.values():
            for key in seats_to_release:
                if key in show.seats:
                    show.seats[key].is_selected = False
                    self.log(f"Released seat {key} back to inventory")

    async def handle_event(self, event: Event):
        pass

# ==================== BOOKMYSHOW SIMULATION ====================

class BookMyShow:
    """Main application simulating BookMyShow platform"""

    def __init__(self):
        self.bus = MessageBus()
        self.shows_db = self._init_database()

        # Initialize agents
        self.availability_agent = AvailabilityAgent(self.bus, self.shows_db)
        self.seat_agent = SeatSelectionAgent(self.bus, self.shows_db)
        self.payment_agent = PaymentAgent(self.bus)
        self.orchestrator = BookingOrchestrator(self.bus, self.shows_db)

        self.print_banner()

    def _init_database(self) -> Dict[str, Show]:
        """Initialize mock database with shows"""
        shows = {}

        # Create a sample movie
        movie = Movie(
            id="mv001",
            title="Dune: Part Three",
            language="English",
            format="IMAX",
            duration=165,
            rating=4.8
        )

        # Create theater layout
        show = Show(
            id="sh001",
            movie=movie,
            theater="PVR Phoenix Marketcity",
            screen="Screen 4 IMAX",
            start_time=datetime(2026, 5, 17, 19, 30)
        )

        # Generate seats: Platinum (A-B), Gold (C-F), Silver (G-J)
        categories = {
            'A': 'PLATINUM', 'B': 'PLATINUM',
            'C': 'GOLD', 'D': 'GOLD', 'E': 'GOLD', 'F': 'GOLD',
            'G': 'SILVER', 'H': 'SILVER', 'I': 'SILVER', 'J': 'SILVER'
        }
        prices = {'PLATINUM': 500, 'GOLD': 350, 'SILVER': 200}

        for row in categories:
            for num in range(1, 11):
                key = f"{row}{num}"
                show.seats[key] = Seat(
                    row=row,
                    number=num,
                    category=categories[row],
                    price=prices[categories[row]],
                    is_available=random.random() > 0.2  # 80% availability
                )

        shows[show.id] = show
        return shows

    def print_banner(self):
        print("=" * 70)
        print("🎬  BOOKMYSHOW MULTI-AGENT TICKET BOOKING SYSTEM")
        print("=" * 70)
        print("Agents: Availability | Seat Selection | Payment | Orchestrator")
        print("-" * 70)

    def print_theater_layout(self, show_id: str):
        """Visual representation of theater seats"""
        show = self.shows_db[show_id]
        print(f"\n📍 {show.theater} - {show.screen}")
        print(f"🎬 {show.movie.title} | {show.movie.format} | {show.start_time.strftime('%I:%M %p')}")
        print("\n         🎭 SCREEN 🎭")
        print("-" * 40)

        for row in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']:
            seats_row = []
            for num in range(1, 11):
                seat = show.seats[f"{row}{num}"]
                if seat.is_selected:
                    seats_row.append("🟡")
                elif seat.is_available:
                    seats_row.append("🟢")
                else:
                    seats_row.append("🔴")
            cat = show.seats[f"{row}1"].category
            print(f"  {row} ({cat:8}) {' '.join(seats_row)}")

        print("\n🟢 Available  🟡 Selected  🔴 Booked")

    async def book_tickets(self, movie: str, theater: str, num_tickets: int = 2,
                          category: str = "GOLD", user_email: str = "user@example.com") -> str:
        """Public API to book tickets"""
        request = BookingRequest(
            movie_title=movie,
            theater=theater,
            num_tickets=num_tickets,
            preferred_category=category,
            user_email=user_email
        )

        print(f"\n{'='*50}")
        print(f"🎫 NEW BOOKING REQUEST: {request.id}")
        print(f"   Movie: {movie}")
        print(f"   Theater: {theater}")
        print(f"   Tickets: {num_tickets} ({category})")
        print(f"{'='*50}")

        booking_id = await self.orchestrator.start_booking(request)

        # Wait for completion (with timeout)
        for _ in range(30):  # 30 seconds max
            await asyncio.sleep(0.5)
            if booking_id in self.orchestrator.completed_bookings:
                return booking_id
            if booking_id in self.orchestrator.failed_bookings:
                return booking_id

        return booking_id

    def get_booking_status(self, booking_id: str) -> Optional[BookingRequest]:
        """Get current status of a booking"""
        if booking_id in self.orchestrator.completed_bookings:
            return self.orchestrator.completed_bookings[booking_id]
        if booking_id in self.orchestrator.failed_bookings:
            return self.orchestrator.failed_bookings[booking_id]
        if booking_id in self.orchestrator.active_bookings:
            return self.orchestrator.active_bookings[booking_id]
        return None

    def print_booking_summary(self, booking_id: str):
        """Print detailed booking summary"""
        booking = self.get_booking_status(booking_id)
        if not booking:
            print(f"❌ Booking {booking_id} not found")
            return

        print(f"\n{'='*60}")
        print(f"📋 BOOKING SUMMARY: {booking_id}")
        print(f"{'='*60}")
        print(f"Status: {booking.status.name}")
        print(f"Movie: {booking.movie_title}")
        print(f"Theater: {booking.theater}")
        print(f"Seats: {[f'{s.row}{s.number}' for s in booking.selected_seats]}")
        print(f"Amount: ₹{booking.total_amount}")
        if booking.error_message:
            print(f"Error: {booking.error_message}")
        print(f"\n📜 Audit Log:")
        for log in booking.audit_log:
            print(f"   {log}")
        print(f"{'='*60}")

# ==================== DEMO EXECUTION ====================

async def main():
    # Initialize platform
    bms = BookMyShow()

    # Show initial theater layout
    bms.print_theater_layout("sh001")

    # Scenario 1: Successful booking
    print("\n" + "🔥"*35)
    print("SCENARIO 1: Standard Booking Flow")
    print("🔥"*35)

    booking1 = await bms.book_tickets(
        movie="Dune: Part Three",
        theater="PVR Phoenix Marketcity",
        num_tickets=3,
        category="GOLD"
    )

    await asyncio.sleep(2)  # Let async operations complete
    bms.print_booking_summary(booking1)
    bms.print_theater_layout("sh001")

    # Scenario 2: Failed booking (insufficient seats or payment failure)
    print("\n" + "🔥"*35)
    print("SCENARIO 2: Handling Failure (Payment/Availability)")
    print("🔥"*35)

    booking2 = await bms.book_tickets(
        movie="Dune: Part Three",
        theater="PVR Phoenix Marketcity",
        num_tickets=5,
        category="PLATINUM"
    )

    await asyncio.sleep(2)
    bms.print_booking_summary(booking2)

    # Scenario 3: Concurrent bookings (race condition simulation)
    print("\n" + "🔥"*35)
    print("SCENARIO 3: Concurrent Bookings (Race Condition Test)")
    print("🔥"*35)

    tasks = [
        bms.book_tickets("Dune: Part Three", "PVR Phoenix Marketcity", 2, "SILVER", "user1@test.com"),
        bms.book_tickets("Dune: Part Three", "PVR Phoenix Marketcity", 2, "SILVER", "user2@test.com"),
        bms.book_tickets("Dune: Part Three", "PVR Phoenix Marketcity", 2, "SILVER", "user3@test.com"),
    ]

    results = await asyncio.gather(*tasks)
    await asyncio.sleep(3)

    for bid in results:
        bms.print_booking_summary(bid)

    # Final theater state
    print("\n" + "="*60)
    print("FINAL THEATER STATE")
    print("="*60)
    bms.print_theater_layout("sh001")

    # Print event history
    print(f"\n📊 Total Events Processed: {len(bms.bus.event_history)}")
    print("Event Flow:")
    for i, event in enumerate(bms.bus.event_history[-10:], 1):  # Last 10 events
        print(f"   {i}. {event.type.value} (from {event.source})")

if __name__ == "__main__":
    nest_asyncio.apply()
    asyncio.run(main())

🎬  BOOKMYSHOW MULTI-AGENT TICKET BOOKING SYSTEM
Agents: Availability | Seat Selection | Payment | Orchestrator
----------------------------------------------------------------------

📍 PVR Phoenix Marketcity - Screen 4 IMAX
🎬 Dune: Part Three | IMAX | 07:30 PM

         🎭 SCREEN 🎭
----------------------------------------
  A (PLATINUM) 🟢 🟢 🟢 🟢 🔴 🟢 🟢 🔴 🟢 🟢
  B (PLATINUM) 🔴 🟢 🔴 🟢 🟢 🟢 🟢 🟢 🟢 🟢
  C (GOLD    ) 🟢 🟢 🟢 🔴 🟢 🟢 🟢 🟢 🟢 🟢
  D (GOLD    ) 🟢 🟢 🟢 🔴 🟢 🟢 🟢 🟢 🟢 🔴
  E (GOLD    ) 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🔴
  F (GOLD    ) 🔴 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢
  G (SILVER  ) 🟢 🔴 🟢 🟢 🔴 🔴 🔴 🟢 🟢 🟢
  H (SILVER  ) 🟢 🟢 🟢 🔴 🟢 🔴 🟢 🔴 🟢 🔴
  I (SILVER  ) 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🔴
  J (SILVER  ) 🟢 🟢 🟢 🟢 🟢 🔴 🟢 🟢 🔴 🟢

🟢 Available  🟡 Selected  🔴 Booked

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
SCENARIO 1: Standard Booking Flow
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

🎫 NEW BOOKING REQUEST: 47dc1d81
   Movie: Dune: Part Three
   Theater: PVR Phoenix Marketcity
   Tickets: 3 (GOLD)
  🤖 [BookingOrchestrator] Starting new booking flow: 47dc1d81

📡 EVEN